ChatGPT advised that the primary prediction unit is O*NET-SOC Code. I will look for CSVs containing those first.

Instructions labeled the following as important dataframes.

In [1]:
import pandas as pd

In [2]:
df_occupation_data = pd.read_csv('ONET data/Occupation Data.csv')
df_task_statements = pd.read_csv('ONET data/Task Statements.csv')
df_skills = pd.read_csv('ONET data/Skills.csv')
df_abilities = pd.read_csv('ONET data/Abilities.csv')
df_work_activities = pd.read_csv('ONET data/Work Activities.csv')
df_knowledge = pd.read_csv('ONET data/Knowledge.csv')
df_work_context = pd.read_csv('ONET data/Work Context.csv')
df_tools_used = pd.read_csv('ONET data/Tools Used.csv')
df_technology_skills = pd.read_csv('ONET data/Technology Skills.csv')

In [3]:
dfs = {
    "occupation_data": df_occupation_data,
    "task_statements": df_task_statements,
    "skills": df_skills,
    "abilities": df_abilities,
    "work_activities": df_work_activities,
    "knowledge": df_knowledge,
    "work_context": df_work_context,
    "tools_used": df_tools_used,
    "technology_skills": df_technology_skills
}

In [4]:
df_technology_skills['O*NET-SOC Code'].nunique()

923

In [12]:
df_occupation_data['O*NET-SOC Code'].nunique()

1016

There is a mismatch likely resulting from some jobs being absent from some datasets. This could make aggregation more complicated. Let's see what jobs are missing from where.

# Find Missing

In [152]:
soc_sets = {
    name: set(df['O*NET-SOC Code'].unique())
    for name, df in dfs.items()
}


In [153]:
all_socs = sorted(set.union(*soc_sets.values()))


In [154]:
import pandas as pd

coverage_df = pd.DataFrame({
    name: [soc in soc_sets[name] for soc in all_socs]
    for name in soc_sets
}, index=all_socs)

coverage_df.index.name = 'O*NET-SOC Code'
coverage_df = coverage_df.astype(int)

coverage_df['MISSING_COUNT'] = (
    len(coverage_df.columns) - coverage_df.sum(axis=1)
)

coverage_df

,occupation_data,task_statements,skills,abilities,work_activities,knowledge,work_context,tools_used,technology_skills,MISSING_COUNT
O*NET-SOC Code,,,,,,,,,,
11-1011.00,1,1,1,1,1,1,1,1,1,0
11-1011.03,1,1,1,1,1,1,1,1,1,0
11-1021.00,1,1,1,1,1,1,1,1,1,0
11-1031.00,1,1,0,0,0,0,0,1,1,5
11-2011.00,1,1,1,1,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...
55-3014.00,1,0,0,0,0,0,0,0,0,8
55-3015.00,1,0,0,0,0,0,0,0,0,8
55-3016.00,1,0,0,0,0,0,0,0,0,8


In [155]:
coverage_df['MISSING_COUNT'].value_counts()

MISSING_COUNT
0    887
8     93
5     15
6     14
1      7
Name: count, dtype: int64

In [156]:
data_columns = coverage_df.columns.drop('MISSING_COUNT')

total_socs = len(coverage_df)

for col in data_columns:
    missing = (coverage_df[col] == 0).sum()
    print(f"{col}: {missing} jobs missing out of {total_socs}")

occupation_data: 0 jobs missing out of 1016
task_statements: 93 jobs missing out of 1016
skills: 122 jobs missing out of 1016
abilities: 122 jobs missing out of 1016
work_activities: 122 jobs missing out of 1016
knowledge: 122 jobs missing out of 1016
work_context: 122 jobs missing out of 1016
tools_used: 114 jobs missing out of 1016
technology_skills: 93 jobs missing out of 1016
